In [ ]:
import sqlite3

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Carga de la tabla drivers

drivers = pd.read_csv(
    "../data/drivers.csv",
    sep=",",
    thousands=None,
    decimal=".",
)
drivers.head()

In [ ]:
# Carga de la tabla timesheet
timesheet = pd.read_csv(
    "../data/timesheet.csv",
    sep=",",
    thousands=None,
    decimal=".",
)
timesheet.head()

In [ ]:
# Conexión a la base de datos SQLite y almacena las tablas

connection = sqlite3.connect("../temp/db.sqlite")

drivers.to_sql("drivers_raw", connection, if_exists="replace", index=False)
timesheet.to_sql("timesheet_raw", connection, if_exists="replace", index=False)

In [ ]:
# Crea vistas para las tablas drivers y timesheet

connection.executescript(
    """
    DROP VIEW IF EXISTS drivers;
    DROP VIEW IF EXISTS timesheet;

    CREATE VIEW drivers AS
    SELECT
        driverId AS driver_id,
        name,
        ssn,
        location,
        certified,
        \"wage-plan\" AS wage_plan
    FROM drivers_raw;

    CREATE VIEW timesheet AS
    SELECT
        driverId AS driver_id,
        week,
        \"hours-logged\" AS hours_logged,
        \"miles-logged\" AS miles_logged
    FROM timesheet_raw;
    """
)
connection.commit()

In [ ]:
# Tablas y vistas disponibles en la base de datos

sql = """
SELECT
    name,
    type
FROM sqlite_master
WHERE type IN ('table', 'view')
ORDER BY type, name;
"""

pd.read_sql_query(sql, connection)

In [ ]:
# Cinco primeras filas de la tabla de conductores
sql = """
SELECT *
FROM drivers
LIMIT 5;
"""

pd.read_sql_query(sql, connection)

In [ ]:
# Cinco primeras filas de los registros semanales
sql = """
SELECT *
FROM timesheet
LIMIT 5;
"""

pd.read_sql_query(sql, connection)

In [ ]:
# Media de las horas y las millas de cada conductor
sql = """
SELECT
    driver_id,
    AVG(hours_logged) AS mean_hours_logged,
    AVG(miles_logged) AS mean_miles_logged
FROM timesheet
GROUP BY driver_id
ORDER BY driver_id;
"""

mean_timesheet = pd.read_sql_query(sql, connection)
mean_timesheet.head()

In [ ]:
# Registros con valores por debajo de la media del grupo
sql = """
SELECT
    driver_id,
    week,
    hours_logged,
    miles_logged,
    AVG(hours_logged) OVER (
        PARTITION BY driver_id
    ) AS mean_hours_logged
FROM timesheet
ORDER BY driver_id, week;
"""

timesheet_with_means = pd.read_sql_query(sql, connection)
timesheet_with_means.head()

In [ ]:
sql = """
WITH timesheet_with_means AS (
    SELECT
        driver_id,
        week,
        hours_logged,
        miles_logged,
        AVG(hours_logged) OVER (
            PARTITION BY driver_id
        ) AS mean_hours_logged
    FROM timesheet
)
SELECT
    driver_id,
    week,
    hours_logged,
    miles_logged,
    mean_hours_logged
FROM timesheet_with_means
WHERE hours_logged < mean_hours_logged
ORDER BY driver_id, week;
"""

timesheet_below = pd.read_sql_query(sql, connection)
display(timesheet_below.head(), timesheet_below.tail())

In [ ]:
# Cantidad total de horas y millas de cada conductor
sql = """
SELECT
    driver_id,
    SUM(hours_logged) AS hours_logged,
    SUM(miles_logged) AS miles_logged
FROM timesheet
GROUP BY driver_id
ORDER BY driver_id;
"""

sum_timesheet = pd.read_sql_query(sql, connection)
sum_timesheet.head(10)

In [ ]:
# Valores mínimo y máximo de horas registradas por cada conductor
sql = """
SELECT
    driver_id,
    MIN(hours_logged) AS min_hours_logged,
    MAX(hours_logged) AS max_hours_logged
FROM timesheet
GROUP BY driver_id
ORDER BY driver_id;
"""

hours_range = pd.read_sql_query(sql, connection)
hours_range

In [ ]:
connection.executescript(
    """
    DROP VIEW IF EXISTS driver_summary;

    CREATE VIEW driver_summary AS
    SELECT
        t.driver_id,
        SUM(t.hours_logged) AS hours_logged,
        SUM(t.miles_logged) AS miles_logged,
        d.name
    FROM timesheet AS t
    INNER JOIN drivers AS d
        ON t.driver_id = d.driver_id
    GROUP BY
        t.driver_id,
        d.name;
    """
)
connection.commit()

sql = """
SELECT *
FROM driver_summary
ORDER BY driver_id;
"""

summary = pd.read_sql_query(sql, connection)
summary

In [ ]:
# Almacenamiento de los resultados

summary.to_csv(
    "../submission/summary.csv",
    sep=",",
    header=True,
    index=False,
)

In [ ]:
# Los conductores con mayor cantidad total de millas registradas
sql = """
SELECT
    driver_id,
    hours_logged,
    miles_logged,
    name
FROM driver_summary
ORDER BY miles_logged DESC
LIMIT 10;
"""

top10 = pd.read_sql_query(sql, connection)
top10

In [ ]:
# Creación de un gráfico de barras horizontales
# La columna 'name' pasa a ser el nombre de las filas
top10 = top10.set_index("name")

# Paleta de colores:
#   tab:blue     tab:red       tab:pink
#   tab:orange   tab:purple    tab:gray
#   tab:green    tab:brown     tab:olive
#   tab:cyan
top10["miles_logged"].plot.barh(color="tab:orange", alpha=0.6)

plt.gca().invert_yaxis()

plt.gca().get_xaxis().set_major_formatter(
    matplotlib.ticker.FuncFormatter(lambda x, p: format(int(x), ","))
)

plt.xticks(rotation=90)

plt.gca().spines["left"].set_color("lightgray")
plt.gca().spines["bottom"].set_color("gray")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.savefig("../submission/top10_drivers.png", bbox_inches="tight")

In [ ]:
# Cierre de la conexión a la base de datos
connection.close()